In [1]:
!pip install numpy pandas


In [3]:
# Run this cell first
!pip install --quiet googletrans==4.0.0-rc1 deep-translator gradio gtts pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 11.0.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-

In [4]:
import io, time
import pandas as pd
from googletrans import Translator, LANGUAGES
from deep_translator import GoogleTranslator as DeepGoogle
from gtts import gTTS

translator = Translator()  # googletrans (free/unofficial)
print("googletrans and deep-translator ready.")


googletrans and deep-translator ready.


In [5]:
# Creates a small sample CSV. Change/add sentences as you like.
data = {
    "source_text": [
        "Hello, how are you?",
        "I love programming.",
        "Machine learning is amazing.",
        "Good morning!",
        "Have a nice day!"
    ],
    "source_lang": ["en","en","en","en","en"],
    "target_lang": ["fr","es","de","ur","it"]
}
df = pd.DataFrame(data)
local_csv = "translation_dataset.csv"
df.to_csv(local_csv, index=False)
# If Drive mounted, save copy there too
try:
    df.to_csv(f"{DRIVE_BASE}/translation_dataset.csv", index=False)
    print("Saved a copy to Drive:", f"{DRIVE_BASE}/translation_dataset.csv")
except Exception:
    pass

print("Created CSV:", local_csv)
df


Created CSV: translation_dataset.csv


,source_text,source_lang,target_lang
0,"Hello, how are you?",en,fr
1,I love programming.,en,es
2,Machine learning is amazing.,en,de
3,Good morning!,en,ur
4,Have a nice day!,en,it


In [6]:
def safe_translate(text, src='auto', tgt='en'):
    """Try googletrans first; if it fails, fallback to deep-translator."""
    if not text or str(text).strip()=="":
        return ""
    try:
        # googletrans uses dest and src codes ('auto' allowed)
        res = translator.translate(text, src=src if src!='auto' else 'auto', dest=tgt)
        return res.text
    except Exception as e:
        # fallback using deep-translator
        try:
            return DeepGoogle(source=src if src!='auto' else 'auto', target=tgt).translate(text)
        except Exception as e2:
            return f"ERROR: {e} | fallback error: {e2}"

def tts_bytes(text, lang_code='en'):
    """Return mp3 bytes for Gradio audio; returns None on failure."""
    try:
        buf = io.BytesIO()
        g = gTTS(text=text, lang=lang_code)
        g.write_to_fp(buf)
        buf.seek(0)
        return buf.read()
    except Exception as e:
        return None


In [7]:
import csv, sys

def batch_translate_csv(input_path, output_path, default_src='auto', default_tgt='en', sleep=0.15):
    """
    input_path: CSV must contain 'source_text' and optional 'source_lang' and 'target_lang' columns.
    output_path: path to save CSV with added 'translated_text' column.
    """
    df = pd.read_csv(input_path)
    translated = []
    for i, row in df.iterrows():
        text = str(row.get('source_text', '')).strip()
        src = row.get('source_lang', default_src) or default_src
        tgt = row.get('target_lang', default_tgt) or default_tgt
        # Normalize codes like 'en — English' if provided
        if isinstance(src, str) and ' — ' in src: src = src.split(' — ')[0].strip()
        if isinstance(tgt, str) and ' — ' in tgt: tgt = tgt.split(' — ')[0].strip()
        try:
            tr = safe_translate(text, src=src, tgt=tgt)
        except Exception as e:
            tr = f"ERROR: {e}"
        translated.append(tr)
        # be polite and avoid quick fire requests
        time.sleep(sleep)
        if (i+1) % 20 == 0:
            print(f"Translated {i+1}/{len(df)}")
    df['translated_text'] = translated
    df.to_csv(output_path, index=False)
    return df

# Example usage (change to Drive path if desired)
in_csv = "translation_dataset.csv"
out_csv = "translation_dataset_translated.csv"
print("Translating CSV... this may take a few seconds.")
result = batch_translate_csv(in_csv, out_csv)
try:
    # save to Drive too if available
    result.to_csv(f"{DRIVE_BASE}/{out_csv}", index=False)
    print("Saved translated CSV to Drive:", f"{DRIVE_BASE}/{out_csv}")
except Exception:
    pass
result.head()


Translating CSV... this may take a few seconds.


,source_text,source_lang,target_lang,translated_text
0,"Hello, how are you?",en,fr,Bonjour comment allez-vous?
1,I love programming.,en,es,Me encanta la programación.
2,Machine learning is amazing.,en,de,Maschinelles Lernen ist erstaunlich.
3,Good morning!,en,ur,صبح بخیر!
4,Have a nice day!,en,it,Buona giornata!


In [9]:
import gradio as gr
import tempfile

# Build friendly language dropdown options
LANG_OPTIONS = sorted([f"{code} — {LANGUAGES[code].capitalize()}" for code in LANGUAGES])
LANG_OPTIONS = ["auto — Auto"] + LANG_OPTIONS  # include auto-detect at top


def ui_translate(text, src_option, tgt_option, tts_flag=False):
    src_code = src_option.split(" — ")[0]
    tgt_code = tgt_option.split(" — ")[0]
    translated = safe_translate(text, src=src_code, tgt=tgt_code)

    audio_path = None
    if tts_flag:
        try:
            # save temporary mp3 file for playback
            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
            tts = gTTS(text=translated, lang=tgt_code if tgt_code != "auto" else "en")
            tts.save(tmp.name)
            audio_path = tmp.name
        except Exception as e:
            audio_path = None

    return translated, audio_path


with gr.Blocks() as demo:
    gr.Markdown("# 🌍 Free Translation Tool (GoogleTrans + Fallback)")

    with gr.Row():
        txt = gr.Textbox(lines=4, label="Source text", placeholder="Type text to translate...")

    with gr.Row():
        src = gr.Dropdown(LANG_OPTIONS, value="auto — Auto", label="Source language")
        tgt = gr.Dropdown(LANG_OPTIONS, value="en — English", label="Target language")

    tts_opt = gr.Checkbox(label="🔊 Text-to-Speech (optional)", value=False)

    with gr.Row():
        out_text = gr.Textbox(lines=4, label="Translated text")
        out_audio = gr.Audio(label="TTS audio", type="filepath")  # ✅ fixed type

    btn = gr.Button("Translate")
    btn.click(fn=ui_translate, inputs=[txt, src, tgt, tts_opt], outputs=[out_text, out_audio])

    gr.Markdown("Uses free unofficial translation libs. May fail sometimes; for production use official APIs.")

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.1, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://c5f620df436f32be89.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
